# 따릉이 연도별 병합 + 자치구 매핑

**한 번 실행 = 한 해(1월~12월) 병합 후 종료.** 아래 `YEAR`만 바꿔서 연도별로 6번 실행하세요.

**출력 컬럼:** 대여일시, 대여 대여소명, 반납일시, 반납 대여소명, 대여자치구, 반납 자치구, **대여요일**(대여일시 날짜에서 계산: 월~일)
**저장 위치:** `data_save/` 폴더에 **월별 12개 파일** (예: 2022년 1월 → `2201_merge.csv`, 12월 → `2212_merge.csv`)

```
data_clean/                                  <- VSCode에서 이 폴더를 열고 실행
├─ merge_ddareungi.ipynb
├─ data_save/                                <- 결과 저장 (2001_merge.csv ~ 2012_merge.csv ...)
├─ 공공자전거_대여소별_이용정보_20~25/        <- 월별 대여소 정보 (자치구 매핑용, 하위폴더 포함)
└─ 나경원 전처리/서울특별시 공공자전거 대여이력 정보_2020/cleaned_2001.csv ...
```
위에서부터 셀을 순서대로 실행하세요. (`Shift+Enter`)

## 1. 설정  ← `YEAR`만 바꾸세요

In [21]:
import hashlib
import re
from pathlib import Path

import pandas as pd

# ================= 설정 =================
YEAR = 2020                                             # <- 병합할 연도 (2020 ~ 2025)
CLEAN_DIR = Path("./공공자전거_대여반납이력정보")                        # 월별 이용이력(전처리) CSV (하위 폴더 포함)
INFO_DIR = Path("./공공자전거_자치구별_대여소이용정보")    # 월별 대여소 이용정보 CSV (하위 폴더 포함)
SAVE_DIR = Path("./data_save")                           # 결과 저장 폴더
# ========================================

UNMAPPED_PATH = SAVE_DIR / f"unmapped_stations_{YEAR}.csv"
UNKNOWN = "미확인"


def out_path(year, month):
    """월별 저장 경로: 2022년 1월 -> data_save/2201_merge.csv"""
    return SAVE_DIR / f"{year % 100:02d}{month:02d}_merge.csv"


OUT_COLS = ["대여일시", "대여 대여소명", "반납일시", "반납 대여소명", "대여자치구", "반납 자치구", "대여요일"]

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("현재 작업 폴더:", Path.cwd())
print("이용이력 폴더 존재:", CLEAN_DIR.exists(), "| 대여소정보 폴더 존재:", INFO_DIR.exists())
print(f"병합 대상: {YEAR}년 -> {SAVE_DIR}/{YEAR % 100:02d}01_merge.csv ~ {YEAR % 100:02d}12_merge.csv")

현재 작업 폴더: c:\Users\Admin\Desktop\project\data_clean
이용이력 폴더 존재: True | 대여소정보 폴더 존재: True
병합 대상: 2020년 -> data_save/2001_merge.csv ~ 2012_merge.csv


## 2. 함수 정의

In [22]:
def read_csv_auto(path, **kw):
    """utf-8-sig -> cp949 순서로 인코딩 시도"""
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {path}")


def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


def load_info_maps(info_dir):
    """returns {연-월: (번호->자치구 dict, 이름->자치구 dict)}  (전 기간 월별 파일)"""
    maps = {}
    for p in sorted(info_dir.rglob("*.csv")):
        m = re.search(r"(\d{4})[_-](\d{2})(?!\d)", p.stem)
        if not m:
            continue  # ..._2020_전처리 같은 연도별 파일은 건너뜀
        ym = f"{m.group(1)}-{m.group(2)}"
        if ym in maps:
            print(f"[주의] {ym} 파일이 둘 이상 있음 -> 나중 파일 사용: {p.name}")
        df = read_csv_auto(p)
        gu_col = next(c for c in df.columns if "자치구" in c)
        name_col = next(c for c in df.columns if "대여소" in c and "명" in c)
        s = df[name_col].astype(str)
        tmp = pd.DataFrame({
            "no": pd.to_numeric(s.str.extract(r"^\s*(\d+)\s*\.")[0], errors="coerce"),
            "name": s.str.replace(r"^\s*\d+\s*\.\s*", "", regex=True).str.strip(),
            "gu": df[gu_col].astype(str).str.strip(),
        })
        with_no = tmp.dropna(subset=["no"])
        by_no = dict(zip(with_no["no"].astype(int), with_no["gu"]))
        by_name = dict(zip(tmp["name"], tmp["gu"]))
        maps[ym] = (by_no, by_name)
    return maps


def build_global_maps(maps):
    """전체 기간 보조 맵 (최신 달이 우선)"""
    g_no, g_name = {}, {}
    for ym in sorted(maps):
        g_no.update(maps[ym][0])
        g_name.update(maps[ym][1])
    return g_no, g_name


def assign_gu(df, no_col, name_col, dt_col, maps, g_no, g_name):
    """행의 날짜(연-월)에 맞는 월별 맵으로 자치구를 붙임"""
    ym_series = df[dt_col].astype(str).str[:7]
    result = pd.Series(UNKNOWN, index=df.index, dtype="object")
    for ym, idx in ym_series.groupby(ym_series).groups.items():
        sub = df.loc[idx]
        by_no, by_name = maps.get(ym, ({}, {}))
        names = sub[name_col].astype(str).str.strip()
        gu = sub[no_col].map(by_no)              # 1순위: 같은 달 번호
        gu = gu.fillna(names.map(by_name))       # 2순위: 같은 달 이름
        gu = gu.fillna(names.map(g_name))        # 3순위: 전체 기간 이름
        gu = gu.fillna(sub[no_col].map(g_no))    # 4순위: 전체 기간 번호
        result.loc[idx] = gu.fillna(UNKNOWN)
    return result


WEEKDAY_KR = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}   # 필요하면 "월요일" 등으로 변경


def add_weekday(df, dt_col="대여일시"):
    """대여일시의 날짜(앞 10자리 YYYY-MM-DD)로 요일을 계산 (월~일). 날짜가 이상하면 빈 값"""
    d = pd.to_datetime(df[dt_col].astype(str).str[:10], format="%Y-%m-%d", errors="coerce")
    return d.dt.dayofweek.map(WEEKDAY_KR)


def prepare(df):
    """연도별 컬럼 형식 차이를 통일 (번호 컬럼이 없으면 이름 앞 번호에서 추출)"""
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={"대여대여소번호": "대여 대여소번호", "대여대여소명": "대여 대여소명",
                            "반납 대여소번호": "반납대여소번호", "반납 대여소명": "반납대여소명"})
    for no_col, name_col in (("대여 대여소번호", "대여 대여소명"), ("반납대여소번호", "반납대여소명")):
        names = df[name_col].astype(str)
        pref = names.str.extract(r"^\s*(\d+)\s*\.")[0]
        if no_col not in df.columns:
            df[no_col] = pd.to_numeric(pref, errors="coerce")
        df[name_col] = names.str.replace(r"^\s*\d+\s*\.\s*", "", regex=True).str.strip()
    return df


def pick_year_files(clean_dir, year):
    """cleaned_YYMM*.csv 중 해당 연도 파일을 (월, 경로) 리스트로"""
    yy = f"{year % 100:02d}"
    found = []
    for p in clean_dir.rglob("*.csv"):
        m = re.match(r"cleaned_(\d{2})(\d{2})", p.stem)
        if m and m.group(1) == yy and 1 <= int(m.group(2)) <= 12:
            found.append((int(m.group(2)), p))
    return sorted(found, key=lambda x: (x[0], str(x[1])))

## 3. 대여소 → 자치구 맵 만들기 (`info` 폴더, 전 기간 월별 파일)

In [23]:
maps = load_info_maps(INFO_DIR)
assert maps, f"{INFO_DIR} 에서 월별 대여소 정보 CSV(..._YYYY_MM.csv)를 찾지 못했어요."
g_no, g_name = build_global_maps(maps)

print(f"대여소 정보 {len(maps)}개월 로드: {min(maps)} ~ {max(maps)}")
need = [f"{YEAR}-{m:02d}" for m in range(1, 13)]
miss = [ym for ym in need if ym not in maps]
print(f"{YEAR}년 대여소 정보 누락 월:", miss if miss else "없음 (12개월 모두 있음)")

대여소 정보 72개월 로드: 2020-01 ~ 2025-12
2020년 대여소 정보 누락 월: 없음 (12개월 모두 있음)


## 4. 이번에 병합할 파일 확인

In [24]:
year_files = pick_year_files(CLEAN_DIR, YEAR)
assert year_files, f"{CLEAN_DIR} 안에서 cleaned_{YEAR % 100:02d}MM*.csv 를 찾지 못했어요."

for mm, p in year_files:
    print(f"{YEAR}-{mm:02d}  {p}")
months_found = sorted({mm for mm, _ in year_files})
miss_m = [m for m in range(1, 13) if m not in months_found]
print(f"\n찾은 월: {months_found}")
print("빠진 월:", miss_m if miss_m else "없음 (12개월 모두 있음)")

2020-01  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2001.csv
2020-02  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2002.csv
2020-03  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2003.csv
2020-04  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2004.csv
2020-05  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2005.csv
2020-06  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2006.csv
2020-07  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2007.csv
2020-08  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2008.csv
2020-09  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2009.csv
2020-10  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2010.csv
2020-11  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2011.csv
2020-12  공공자전거_대여반납이력정보\서울특별시 공공자전거 대여이력 정보_2020\cleaned_2012.csv

찾은 월: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
빠진 월: 없음 (12개월 모두 있음)


## 5. 병합 실행 → `data_save/YYMM_merge.csv` (월별)
월별 파일을 1월부터 차례로 읽어 자치구를 붙이고, **월마다 따로** 저장합니다.
- 내용이 완전히 같은 중복 파일은 건너뜁니다.
- **파일명 월과 데이터 안의 날짜 월이 다르면 합치지 않고 `[제외]`로 알려줍니다.**

In [25]:
# 이번 연도의 기존 결과 파일은 지우고 새로 만듦 (다른 연도 파일은 건드리지 않음)
for m in range(1, 13):
    if out_path(YEAR, m).exists():
        out_path(YEAR, m).unlink()

written = set()
seen_hash, unmapped_rows, failed, excluded, summary, total = {}, [], [], [], [], 0
for mm, p in year_files:
    expected_ym = f"{YEAR}-{mm:02d}"
    h = file_md5(p)
    if h in seen_hash:
        print(f"[중복 파일 건너뜀] {p.name} == {seen_hash[h]}")
        continue
    seen_hash[h] = p.name

    try:
        df = prepare(read_csv_auto(p))

        # 파일명 월 vs 실제 데이터 월 검사
        main_ym = df["대여일시"].astype(str).str[:7].mode().iloc[0]
        if main_ym != expected_ym:
            msg = f"파일명은 {expected_ym}인데 데이터 대부분이 {main_ym}"
            excluded.append((p.name, msg))
            print(f"[제외] {p.name}: {msg}")
            continue

        df["대여자치구"] = assign_gu(df, "대여 대여소번호", "대여 대여소명", "대여일시", maps, g_no, g_name)
        df["반납 자치구"] = assign_gu(df, "반납대여소번호", "반납대여소명", "반납일시", maps, g_no, g_name)
        df = df.rename(columns={"반납대여소명": "반납 대여소명"})
        df["대여요일"] = add_weekday(df, "대여일시")
        n_bad_dt = int(df["대여요일"].isna().sum())
        if n_bad_dt:
            print(f"[주의] {p.name}: 날짜를 읽지 못해 요일이 빈 행 {n_bad_dt:,}개")

        out = df[OUT_COLS]
        first = mm not in written          # 같은 달 파일이 둘 이상이면 이어 붙임
        out.to_csv(out_path(YEAR, mm), mode="w" if first else "a", header=first,
                   index=False, encoding="utf-8-sig")
        written.add(mm)

        n_r = int((out["대여자치구"] == UNKNOWN).sum())
        n_b = int((out["반납 자치구"] == UNKNOWN).sum())
        print(f"{p.name} -> {out_path(YEAR, mm).name}: {len(out):,}행 | 미확인 대여 {n_r:,} / 반납 {n_b:,}")
        summary.append({"월": expected_ym, "저장 파일": out_path(YEAR, mm).name, "행수": len(out), "미확인 대여": n_r, "미확인 반납": n_b})
        total += len(out)

        bad = df[df["대여자치구"] == UNKNOWN]
        if len(bad):
            u = (bad.assign(연월=bad["대여일시"].astype(str).str[:7])
                    .groupby(["연월", "대여 대여소번호", "대여 대여소명"]).size()
                    .reset_index(name="건수"))
            unmapped_rows.append(u)
    except Exception as e:
        failed.append((str(p), repr(e)))
        print(f"[실패] {p}: {e!r}")

if unmapped_rows:
    pd.concat(unmapped_rows).to_csv(UNMAPPED_PATH, index=False, encoding="utf-8-sig")
    print(f"\n자치구를 못 찾은 대여소 목록 -> {UNMAPPED_PATH}")

if total:
    print(f"\n✅ {YEAR}년 완료: {len(written)}개월 파일 저장, 총 {total:,}행 -> {SAVE_DIR}/")
else:
    print(f"\n⚠ {YEAR}년 결과 파일이 만들어지지 않았어요. 위의 [제외]/[실패] 메시지를 확인하세요.")
missing_out = [f"{YEAR % 100:02d}{m:02d}_merge.csv" for m in range(1, 13) if m not in written]
if total and missing_out:
    print("⚠ 만들어지지 않은 월 파일:", missing_out)
if excluded or failed:
    print("확인 필요:", excluded + failed)

cleaned_2001.csv -> 2001_merge.csv: 360,671행 | 미확인 대여 35 / 반납 71
cleaned_2002.csv -> 2002_merge.csv: 355,204행 | 미확인 대여 40 / 반납 77
cleaned_2003.csv -> 2003_merge.csv: 634,374행 | 미확인 대여 83 / 반납 151
cleaned_2004.csv -> 2004_merge.csv: 829,504행 | 미확인 대여 82 / 반납 94
cleaned_2005.csv -> 2005_merge.csv: 885,244행 | 미확인 대여 61 / 반납 56
cleaned_2006.csv -> 2006_merge.csv: 1,119,110행 | 미확인 대여 123 / 반납 98
cleaned_2007.csv -> 2007_merge.csv: 1,141,633행 | 미확인 대여 219 / 반납 109
cleaned_2008.csv -> 2008_merge.csv: 801,498행 | 미확인 대여 90 / 반납 76
cleaned_2009.csv -> 2009_merge.csv: 1,347,566행 | 미확인 대여 164 / 반납 106
cleaned_2010.csv -> 2010_merge.csv: 1,447,191행 | 미확인 대여 183 / 반납 149
cleaned_2011.csv -> 2011_merge.csv: 1,018,138행 | 미확인 대여 155 / 반납 141
cleaned_2012.csv -> 2012_merge.csv: 628,113행 | 미확인 대여 104 / 반납 118

자치구를 못 찾은 대여소 목록 -> data_save\unmapped_stations_2020.csv

✅ 2020년 완료: 12개월 파일 저장, 총 10,568,246행 -> data_save/


## 6. 월별 요약 / 저장된 파일 확인

In [26]:
if summary:
    s = pd.DataFrame(summary)
    s["미확인 대여(%)"] = (s["미확인 대여"] / s["행수"] * 100).round(2)
    s["미확인 반납(%)"] = (s["미확인 반납"] / s["행수"] * 100).round(2)
    display(s)

    first_file = out_path(YEAR, min(written))
    chk = pd.read_csv(first_file, encoding="utf-8-sig", nrows=200_000)   # 큰 파일이라 앞부분만
    print(f"[{first_file.name}] 컬럼:", chk.columns.tolist())
    print("빈 값(앞 20만 행):", chk.isna().sum().to_dict())
    display(chk.head())
    print("요일 분포(앞 20만 행):", chk["대여요일"].value_counts().to_dict())
else:
    print("요약할 결과가 없어요.")

,월,저장 파일,행수,미확인 대여,미확인 반납,미확인 대여(%),미확인 반납(%)
0,2020-01,2001_merge.csv,360671,35,71,0.01,0.02
1,2020-02,2002_merge.csv,355204,40,77,0.01,0.02
2,2020-03,2003_merge.csv,634374,83,151,0.01,0.02
3,2020-04,2004_merge.csv,829504,82,94,0.01,0.01
4,2020-05,2005_merge.csv,885244,61,56,0.01,0.01
5,2020-06,2006_merge.csv,1119110,123,98,0.01,0.01
6,2020-07,2007_merge.csv,1141633,219,109,0.02,0.01
7,2020-08,2008_merge.csv,801498,90,76,0.01,0.01
8,2020-09,2009_merge.csv,1347566,164,106,0.01,0.01
9,2020-10,2010_merge.csv,1447191,183,149,0.01,0.01


[2001_merge.csv] 컬럼: ['대여일시', '대여 대여소명', '반납일시', '반납 대여소명', '대여자치구', '반납 자치구', '대여요일']
빈 값(앞 20만 행): {'대여일시': 0, '대여 대여소명': 0, '반납일시': 0, '반납 대여소명': 0, '대여자치구': 0, '반납 자치구': 0, '대여요일': 0}


,대여일시,대여 대여소명,반납일시,반납 대여소명,대여자치구,반납 자치구,대여요일
0,2020-01-01 0:01,삼부르네상스파크빌,2020-01-01 0:05,디지털입구 교차로,구로구,구로구,수
1,2020-01-01 0:03,대흥역 1번출구,2020-01-01 0:05,서강대 후문 옆,마포구,마포구,수
2,2020-01-01 0:01,원묵고등학교,2020-01-01 0:06,먹골역 1번출구 뒤,중랑구,중랑구,수
3,2020-01-01 0:04,서울 월드컵 경기장,2020-01-01 0:06,마포구청 앞,마포구,마포구,수
4,2020-01-01 0:02,영등포구청역 7번출구,2020-01-01 0:07,양평동6차현대아파트 앞,영등포구,영등포구,수


요일 분포(앞 20만 행): {'금': 40423, '목': 38965, '토': 32777, '수': 29927, '일': 24865, '월': 19105, '화': 13938}


## 7. 자치구를 못 찾은 대여소 (건수 많은 순)

In [27]:
if UNMAPPED_PATH.exists() and summary:
    um = pd.read_csv(UNMAPPED_PATH, encoding="utf-8-sig")
    display(um.groupby(["대여 대여소번호", "대여 대여소명"])["건수"].sum()
              .sort_values(ascending=False).head(30))
else:
    print("못 찾은 대여소 없음")

대여 대여소번호  대여 대여소명
10        위트콤        379
9999      상담센터       268
9997      이수센터       194
3         중랑센터       192
99999     영남센터       128
5         상암센터       108
9993      개화센터        20
9998      영남센터        14
9992      사대문센터       13
9990      _도봉센터        8
9985      천왕센터         6
9988      천호센터         4
9989      종묘센터         2
9991      훈련원센터        2
99998     상암센터         1
Name: 건수, dtype: int64